### Setup

In [ ]:

# This is a demo notebook that simulates some sersics at different redshifts. There hasn't been an effort to seriously fit things yet and iteration counts are seriously reduced,
# just ensuring that things work and demonstrating how to integrate the files in /wip/ into your projects

import sys
sys.path.append('../../src')

In [ ]:
# optionally, enable jax computation cache, though not recommended for rapidly changing code
#import os
#comp_cache_path = f"{os.environ['SCRATCH']}/jax_comp_cache"
#os.system(f"mkdir {comp_cache_path}")
#jax.config.update("jax_compilation_cache_dir", comp_cache_path)
#jax.config.update("jax_persistent_cache_enable_xla_caches", "all")

In [ ]:
#version tracking - currently should be jax==0.10.0, cuda==13.0 for compatibility branch
import jax
print(f"jax version = {jax.__version__}")
!nvidia-smi
print(f"devices: {jax.devices()}")
print(f"platform: {jax.devices()[0].platform}") # even if nvidia-smi displays the GPUs, this and the devices will verify jaxlib linked to cuDNN correctly. it should show 'gpu'

In [ ]:
#checking which install we are actually pointing to 
import gigalens
print(gigalens.__path__[0])

In [ ]:
# pre-import module hacking
# when importing, we don't want the classes to even load the older modules, so we define them now
# the first time reloading these here and in the next two cells takes a very long time in my experience, will figure it out later
from importlib import reload
sys.path.append("../../wip")

import new_simulator
reload(new_simulator)
sys.modules["gigalens.jax.simulator"] = new_simulator

import new_shapelets
reload(new_shapelets)
sys.modules["gigalens.jax.profiles.light.shapelets"] = new_shapelets


In [ ]:
#imports
from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.jax.physical_model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.cosmo import w0waCDM_Cosmo as Cosmo
from gigalens.jax.profiles import mass, light
from gigalens.jax.prior import Prior, make_prior_and_model

import jax
import optax
from jax import numpy as jnp
import numpy as np

import tensorflow_probability.substrates.jax as tfp
from tensorflow_probability.substrates.jax import distributions as tfd
from corner import corner
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
# now we supplant things, in a reloadable way
# you shouldn't need to reload anything inside of /src/, but we know it isn't fully stable right now
# if there's any issue, try restarting the kernel - this code may not be 100% reliable, messing with module references is finnicky

def update_modules():
    
    reload(gigalens.simulator)
    import new_simulator
    reload(new_simulator)
    sys.modules["gigalens.jax.simulator"] = new_simulator

    import new_shapelets
    reload(new_shapelets)
    sys.modules["gigalens.jax.profiles.light.shapelets"] = new_shapelets

    global LensSimulator, SimulatorConfig, ModellingSequence # module saving is weird like that

    reload(gigalens.jax.inference)
    from gigalens.jax.inference import ModellingSequence
    
    import mclmc_alt
    reload(mclmc_alt)
    ModellingSequence.MCLMC = mclmc_alt.MCLMC
    
    import nuts
    reload(nuts)
    ModellingSequence.NUTS = nuts.NUTS
    
    import gmm
    reload(gmm)
    ModellingSequence.GMM = gmm.GMM

    LensSimulator = new_simulator.LensSimulator
    SimulatorConfig = gigalens.simulator.SimulatorConfig 
    ModellingSequence = gigalens.jax.inference.ModellingSequence


update_modules()

In [ ]:
# set cosmology constants
z_lens = 0.5
z_s1 = 1.2
z_s2_truth = 2.3  # we want to recover it
Om0_truth = 0.3
w0_truth = -1.0

In [ ]:
# define prior
epl_prior = Prior(
            mass.epl.EPL(), # the profile
            dict(  # the prior
                theta_E=tfd.LogNormal(jnp.log(1.25), 0.25),
                gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
                e1=tfd.Normal(0, 0.1),
                e2=tfd.Normal(0, 0.1),
                center_x=0.1,  # the position is fixed
                center_y=0.0,
            )
)
shear_prior = Prior(
            mass.shear.Shear(),
            dict(gamma1=tfd.Normal(0, 0.05), 
                 gamma2=tfd.Normal(0, 0.05)
                )
)
lens_light_prior = Prior(
            light.sersic.SersicEllipse(),
            dict(
                R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15),
                n_sersic=tfd.Uniform(2, 6),
                e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                center_x=0.1,
                center_y=0.0,
                Ie=tfd.LogNormal(jnp.log(500.0), 0.3),
            )
)

source_1_prior = Prior(
            light.sersic.SersicEllipse(),
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15),
                n_sersic=tfd.Uniform(0.5, 4),
                e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.25),
                center_y=tfd.Normal(0, 0.25),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.5),
                z_source=z_s1
            )
)
source_2_prior = Prior(
            light.sersic.SersicEllipse(),
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15),
                n_sersic=tfd.Uniform(0.5, 4),
                e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.25),
                center_y=tfd.Normal(0, 0.25),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.5),
                z_source=tfd.Uniform(1.2, 3.5)  # unknown redshift
            )
)
cosmo_prior = Prior(
        Cosmo(z_lens=z_lens, z_source_ref=z_s1),  # you need to set the redshifts for the cosmology to work, theta_E is relative to z_source_ref
    dict(
        H0=70.,
        Om0=tfd.Uniform(0.01, 0.99),  # free
        w0=tfd.Uniform(-2, -1/3),  # free
        wa=0.0,
        k=0.0,
    )
)

In [ ]:
# make prior
prior, phys_model = make_prior_and_model(
    lenses=[
        epl_prior,
        shear_prior,
    ],
    sources=[
        source_1_prior,
        source_2_prior
    ],
    foreground=[
        lens_light_prior
    ],
    cosmo=cosmo_prior
)

In [ ]:
# sample from prior for full truth

truth = prior.sample(1, jax.random.PRNGKey(0))
truth['source_light']['1']['z_source'] = z_s2_truth
truth['cosmo']['Om0'] = Om0_truth
truth['cosmo']['w0'] = w0_truth

In [ ]:
# establish sim_config + lens_sim
background_rms=0.1
exp_time=200
gigalens_path = gigalens.__path__[0]
kernel = np.load(f'{gigalens_path}/assets/psf.npy').astype(np.float32)

def update_sim():
    sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
    lens_sim = LensSimulator(phys_model, sim_config, bs=1)
    return sim_config, lens_sim

# unfortunate assumption, that phys_model will not need to be reloaded. will fix soon
sim_config, lens_sim = update_sim()

In [ ]:
# create observed image from truth_sim
truth_sim = LensSimulator(phys_model, sim_config, bs=1)
simulated = truth_sim.simulate(truth)
err_map = np.sqrt(background_rms**2 + np.clip(simulated, 0, np.inf)/exp_time)
np.random.seed(1)
observed_img = simulated + np.random.normal(scale=err_map)

plt.imshow(observed_img, vmax=10)
plt.show()

In [ ]:
# reload modules and objects, and create prob_model and model_seq
# run this cell or this function to on-the-fly redefine your 

#not currently being changed
def update_prob_model():
    return ForwardProbModel(prior, observed_img, background_rms=background_rms, exp_time=exp_time, 
                              include_pixels=True, include_positions=False)

prob_model = update_prob_model()

def update_model_seq():
    update_modules()
    sim_config, _ = update_sim()
    model_seq = ModellingSequence(phys_model, prob_model, sim_config)
    return model_seq
    

model_seq = update_model_seq()

### Fitting:

#### MAP

In [ ]:
# MAP
model_seq = update_model_seq()
opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
jax.clear_caches()
best_model, best_lp, map_chisqs = model_seq.MAP(opt, seed=42,num_steps=1000)

In [ ]:
plt.plot(map_chisqs)
plt.ylim(0.7*np.min(map_chisqs),10.0)

In [ ]:
best_p = prob_model.bij.forward(best_model.tolist())
param_list = [' '.join([ii.key for ii in i]) for i in list(map(list, zip(*jax.tree.flatten_with_path(best_p)[0])))[0]]

In [ ]:
simulator = LensSimulator(phys_model, sim_config, bs=1)
simulated = simulator.simulate(best_p)

plt.figure(figsize=(22, 6))

plt.subplot(131)
plt.imshow(observed_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=10), origin="lower")
plt.colorbar()
plt.axis('off')

plt.subplot(132)
plt.imshow(simulated, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=10), origin="lower")
plt.colorbar()
plt.axis('off')

plt.subplot(133)
resid = observed_img - simulated
err_map = np.sqrt(simulated / exp_time + background_rms**2)
plt.imshow(resid/err_map, cmap='coolwarm', interpolation='none', vmin=-4, vmax=4, origin="lower")
plt.colorbar()
plt.axis('off')

plt.tight_layout()

plt.show()

#### SVI + GMM

In [ ]:
# SVI
model_seq = update_model_seq()
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
jax.clear_caches()
qz, loss_hist = model_seq.SVI(best_model, opt, n_vi=400, num_steps=5000)

In [ ]:
# GMM
model_seq = update_model_seq()
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
jax.clear_caches()

# ensure n_vi // n_gaussians >= dev_cnt 
# it is being evil again, the pmean is reintroducing noise and causing issues
n_gaussians =40
qz_gmm, loss_hist_gmm = model_seq.GMM(best_model, opt, n_vi=4000, num_steps=5000, alpha = 0.7, T=n_gaussians, weight_threshold = 0.01, spread_scales = 0.01)
qz_gmm.covariance = lambda : jnp.mean(qz_gmm.components_distribution.covariance(),axis=0) #this has been helpful in the past, once DP part 2 is done it should go back to normal

In [ ]:
# plot results
fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].plot(loss_hist)
#ax[0].set_ylim(np.min(loss_hist_gmm),np.max(loss_hist))
ax[0].set_title("SVI -ELBO")
ax[1].plot(loss_hist_gmm, color='red')
#ax[1].set_ylim(np.min(loss_hist_gmm),np.max(loss_hist))
ax[1].set_title(rf"SVI_GMM -ELBO $n = {n_gaussians}$")
ax[1].set_yscale("symlog")
plt.show()

In [ ]:
# corner plotting fn
def get_corner_samples(physical, key):
    """
    Key tells you lens_mass (1), lens_light (2), or source_light (3)
    """
    return_list = []

    if key == 1 or 1 in key:
        for i in physical[0][0].keys():
            return_list.append(physical[0][0][i])
        for j in physical[0][1].keys():
            return_list.append(physical[0][1][j])

    if key == 2 or 2 in key:
        for i in physical[1][0].keys():
            return_list.append(physical[1][0][i])

    if key == 3 or 3 in key:
        for i in physical[2][0].keys():
            return_list.append(physical[2][0][i])
            
    return np.stack(return_list).T


def flatten_samples(samples, components=('lens_mass', 'cosmo', 'redshift')):
    flat_samples = []
    labels = []
    for k0 in components:
        if k0 == 'redshift':
            k0 = 'source_light'
            k2 = 'z_source'
            ds = samples.get(k0, {})
            for k1 in sorted(ds.keys()):
                if k2 in samples[k0][k1]:
                    flat_samples.append(np.asarray(samples[k0][k1][k2].flatten()))
                    labels.append(f"{k2}_{k1}")
        else:
            ds = samples.get(k0, {})
            for k1 in sorted(ds.keys()):
                if k0 != 'cosmo':
                    for k2 in sorted(ds[k1].keys()):
                        flat_samples.append(np.asarray(samples[k0][k1][k2].flatten()))
                        labels.append(f"{k2}_{k1}")
                else:
                    flat_samples.append(np.asarray(samples[k0][k1].flatten()))
                    labels.append(k1)
    return np.stack(flat_samples), labels


def corner_plot(samples, sigma_levels=np.array([1., 2., 3.]), components=('lens_mass', 'cosmo', 'redshift'),color="black",alpha=1.0,fig=True,atch=None):
    corner_samples, labels = flatten_samples(samples, components=components)
    if fig:
        figure = corner(corner_samples.T, bins=20, range=np.ones(len(labels)) * 0.999,
                    plot_datapoints=False, plot_density=False,
                    fill_contours=True, show_titles=True,
                    quantiles=[0.16, 0.50, 0.84],
                    levels=(1.0 - np.exp(-0.5 * sigma_levels ** 2)),
                    labels=labels,
                    title_fmt=".2e",
                    color=color,
                    alpha=alpha,
                    )
        return figure
    else:
        corner(corner_samples.T, bins=20, range=np.ones(len(labels)) * 0.999,
                    plot_datapoints=False, plot_density=False,
                    fill_contours=True, show_titles=True,
                    quantiles=[0.16, 0.50, 0.84],
                    levels=(1.0 - np.exp(-0.5 * sigma_levels ** 2)),
                    labels=labels,
                    title_fmt=".2e",
                    fig = atch,
                    color=color,
                    alpha=alpha,
                    )
        return None


In [ ]:
# corner plot of SVI steps:
n_samples = 10000
samples_sviz = prob_model.bij.forward(list(qz.sample(n_samples, jax.random.PRNGKey(0)).reshape(-1,len(param_list)).T))
samples_gmmz = prob_model.bij.forward(list(qz_gmm.sample(n_samples,jax.random.PRNGKey(0)).reshape(-1,len(param_list)).T))

_ = corner_plot(samples_gmmz,components=('lens_mass','source_light',"cosmo"),color="green")
corner_plot(samples_sviz,components=('lens_mass','source_light',"cosmo"),fig = False,atch=_,color="yellow",alpha=0.15)

#### HMC + MCLMC + NUTS

In [ ]:
# HMC
model_seq = update_model_seq()
jax.clear_caches()
samples_hmc = model_seq.HMC(qz_gmm, num_burnin_steps=1000, num_results=3000)

In [ ]:
# NUTS
model_seq = update_model_seq()
jax.clear_caches()
samples_nuts = model_seq.NUTS(qz, num_burnin_steps=50, num_results=100)

In [ ]:
# MCLMC
reload(gigalens.jax.model)
from gigalens.jax.model import ProbModel, ForwardProbModel
sim_config, lens_sim = update_sim()
model_seq = update_model_seq()
jax.clear_caches()
samples_mclmc = model_seq.MCLMC(qz, num_burnin_steps=150, num_results=1000)

In [ ]:
# rhat, ess calculations
def rhat_and_ess(samps,ind_ndims = 2, cross_dims = [1,2]): 
    return tfp.mcmc.potential_scale_reduction(samps, independent_chain_ndims=ind_ndims), tfp.mcmc.effective_sample_size(samps, cross_chain_dims=cross_dims)

rhat_hmc, ess_hmc = rhat_and_ess(samples_hmc)
rhat_nuts, ess_nuts= rhat_and_ess(jnp.swapaxes(samples_nuts,0,1), ind_ndims=1,cross_dims=[1])
rhat_mclmc, ess_mclmc= rhat_and_ess(samples_mclmc,ind_ndims=1,cross_dims=[1])

print("rhat:")
for i,(rh,rn,rl,par) in enumerate(zip(rhat_hmc, rhat_nuts, rhat_mclmc,param_list)): 
    print(f"{par:<25}  hmc: {rh:.5f} nuts: {rn:.5f}  mclmc: {rl:.5f}")
print("ess:")
for i,(eh,en,el,par) in enumerate(zip(ess_hmc, ess_nuts, ess_mclmc,param_list)): 
    print(f"{par:<25}  hmc: {eh:.5e} nuts: {en:.5e}  mclmc: {el:.5e}")

In [ ]:
# plotting rhats
x_ax = np.arange(0,len(param_list))

plt.scatter(x_ax,(rhat_hmc-1),color='red',label = "HMC")
plt.vlines(x_ax,np.zeros_like(x_ax),(rhat_hmc-1).astype(float),color="red",alpha=0.25) #allows for color and alpha setting, unlike plt.stem()

plt.scatter(x_ax,(rhat_nuts-1),color='blue',label = "NUTS")
plt.vlines(x_ax,np.zeros_like(x_ax),(rhat_nuts-1).astype(float),color="blue",alpha=0.25)

plt.scatter(x_ax,(rhat_mclmc-1),color='green',label = "MCLMC")
plt.vlines(x_ax,np.zeros_like(x_ax),(rhat_mclmc-1).astype(float),color="green",alpha=0.25)

plt.hlines([0.1,0.01],0,len(param_list),color="orange")
plt.title(r"$\hat r - 1$ for each parameter")
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.25, 1), loc="upper right")
plt.xticks(range(len(param_list)),param_list,rotation=90)
plt.show()

In [ ]:
# convert to physical samples
samples_hmcz = prob_model.bij.forward(list(samples_hmc.reshape(-1,len(rhat_hmc)).T))
samples_nutsz = prob_model.bij.forward(list(samples_nuts.reshape(-1,len(rhat_hmc)).T))
samples_mclmcz = prob_model.bij.forward(list(samples_mclmc.reshape(-1,len(rhat_hmc)).T))

In [ ]:
# cosmology corner plot
_ = corner_plot(samples_hmcz,components=('cosmo','redshift'),color="red")
corner_plot(samples_nutsz,components=('cosmo','redshift'),fig = False,atch=_,color="blue",alpha=0.15)
corner_plot(samples_mclmcz,components=('cosmo','redshift'),fig = False,atch=_,color="green",alpha=0.15)

# very interesting plot! NUTS seems to be strongly multimodal, and MCLMC seems to be converging on a smaller region than HMC

In [ ]:
# non-cosmology plot
_ = corner_plot(samples_hmcz,components=('lens_mass','source_light'),color="red")
corner_plot(samples_nutsz,components=('lens_mass','source_light'),fig = False,atch=_,color="blue",alpha=0.15)
corner_plot(samples_mclmcz,components=('lens_mass','source_light'),fig = False,atch=_,color="green",alpha=0.15)

In [ ]:
# full corner plot
_ = corner_plot(samples_hmcz,components=('lens_mass','source_light',"cosmo"),color="red")
corner_plot(samples_nutsz,components=('lens_mass','source_light',"cosmo"),fig = False,atch=_,color="blue",alpha=0.15)
corner_plot(samples_mclmcz,components=('lens_mass','source_light',"cosmo"),fig = False,atch=_,color="green",alpha=0.15)

In [ ]:
# check SVI stuff: 

_ = corner_plot(samples_hmcz,components=('lens_mass','source_light',"cosmo"),color="red")
corner_plot(samples_sviz,components=('lens_mass','source_light',"cosmo"),fig = False,atch=_,color="blue",alpha=0.15)
corner_plot(samples_gmmz,components=('lens_mass','source_light',"cosmo"),fig = False,atch=_,color="green",alpha=0.15)